# 🌾 ANN Classification — Crop Recommendation System
**Domain:** AgriTech | **Model:** Artificial Neural Network (ANN) | **Task:** Multi-Class Classification

---
### 📌 Problem Statement
Given soil nutrient levels and environmental conditions, predict the **most suitable crop** to grow.

### 📦 Dataset Features
| Feature | Description | Unit |
|---------|-------------|------|
| **N** | Nitrogen content in soil | mg/kg |
| **P** | Phosphorus content in soil | mg/kg |
| **K** | Potassium content in soil | mg/kg |
| **temperature** | Average temperature | °C |
| **humidity** | Relative humidity | % |
| **ph** | Soil pH value | 0–14 |
| **rainfall** | Annual rainfall | mm |
| **label** | 🎯 Crop to grow (22 classes) | — |

**Source:** [Kaggle — Crop Recommendation Dataset](https://www.kaggle.com/datasets/atharvaingle/crop-recommendation-dataset)

## 📚 Step 1 — Import Libraries

In [ ]:
# Core
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Visualization
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from matplotlib.gridspec import GridSpec

# Scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, ConfusionMatrixDisplay)

# TensorFlow / Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.utils import to_categorical

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow version : {tf.__version__}")
print(f"NumPy version      : {np.__version__}")
print(f"Pandas version     : {pd.__version__}")

## 📂 Step 2 — Load Dataset

In [ ]:
# -------------------------------------------------------
# Option A: Load from CSV (if you have downloaded it)
# df = pd.read_csv('Crop_recommendation.csv')
# -------------------------------------------------------
# Option B: Auto-generate a realistic synthetic dataset
# (mirrors actual Kaggle dataset statistics)
# -------------------------------------------------------

CROP_PARAMS = {
    'rice':        dict(N=(60,100), P=(30,60),  K=(30,60),  temp=(20,27), hum=(80,90), ph=(5.5,7.0), rain=(150,300)),
    'maize':       dict(N=(60,100), P=(50,80),  K=(50,80),  temp=(18,27), hum=(55,75), ph=(5.5,7.5), rain=(60,110)),
    'chickpea':    dict(N=(0,40),   P=(60,100), K=(60,100), temp=(18,26), hum=(15,25), ph=(5.5,7.5), rain=(60,100)),
    'kidneybeans': dict(N=(0,40),   P=(60,100), K=(60,100), temp=(18,26), hum=(18,28), ph=(5.5,7.5), rain=(100,150)),
    'pigeonpeas':  dict(N=(0,40),   P=(60,80),  K=(60,80),  temp=(18,30), hum=(40,70), ph=(5.0,7.0), rain=(100,150)),
    'mothbeans':   dict(N=(0,40),   P=(40,70),  K=(15,30),  temp=(24,34), hum=(40,60), ph=(3.5,7.0), rain=(30,60)),
    'mungbean':    dict(N=(0,40),   P=(30,60),  K=(30,60),  temp=(25,35), hum=(80,90), ph=(6.2,7.2), rain=(60,100)),
    'blackgram':   dict(N=(0,40),   P=(50,80),  K=(25,50),  temp=(25,35), hum=(60,80), ph=(5.5,7.0), rain=(60,100)),
    'lentil':      dict(N=(0,40),   P=(60,100), K=(60,100), temp=(18,26), hum=(20,40), ph=(6.0,7.5), rain=(40,70)),
    'pomegranate': dict(N=(0,40),   P=(10,30),  K=(30,60),  temp=(18,24), hum=(80,90), ph=(5.5,7.2), rain=(100,150)),
    'banana':      dict(N=(80,120), P=(70,100), K=(40,70),  temp=(25,35), hum=(75,90), ph=(5.5,7.0), rain=(100,150)),
    'mango':       dict(N=(0,40),   P=(15,45),  K=(20,50),  temp=(24,35), hum=(50,70), ph=(5.5,7.5), rain=(90,120)),
    'grapes':      dict(N=(0,40),   P=(100,145),K=(100,145),temp=(8,42),  hum=(80,90), ph=(5.5,7.0), rain=(65,100)),
    'watermelon':  dict(N=(80,120), P=(10,30),  K=(40,60),  temp=(24,35), hum=(80,90), ph=(6.0,7.0), rain=(40,60)),
    'muskmelon':   dict(N=(80,120), P=(10,30),  K=(40,60),  temp=(28,38), hum=(90,95), ph=(6.0,7.5), rain=(20,30)),
    'apple':       dict(N=(0,40),   P=(100,145),K=(130,145),temp=(0,22),  hum=(90,95), ph=(5.5,7.0), rain=(100,125)),
    'orange':      dict(N=(0,20),   P=(5,30),   K=(5,15),   temp=(10,35), hum=(90,95), ph=(6.0,7.5), rain=(100,150)),
    'papaya':      dict(N=(40,80),  P=(50,100), K=(40,80),  temp=(25,35), hum=(90,95), ph=(4.5,7.0), rain=(150,200)),
    'coconut':     dict(N=(0,40),   P=(5,30),   K=(30,60),  temp=(25,35), hum=(90,95), ph=(5.0,8.0), rain=(150,300)),
    'cotton':      dict(N=(100,140),P=(50,100), K=(50,100), temp=(24,40), hum=(55,70), ph=(6.0,8.0), rain=(60,110)),
    'jute':        dict(N=(60,100), P=(40,80),  K=(40,80),  temp=(24,37), hum=(70,90), ph=(6.0,7.0), rain=(150,250)),
    'coffee':      dict(N=(80,120), P=(20,40),  K=(25,45),  temp=(15,28), hum=(50,70), ph=(3.5,7.0), rain=(150,300)),
}

rows = []
for crop, p in CROP_PARAMS.items():
    for _ in range(100):  # 100 samples per crop → 2200 total
        rows.append({
            'N':           np.random.uniform(*p['N']),
            'P':           np.random.uniform(*p['P']),
            'K':           np.random.uniform(*p['K']),
            'temperature': np.random.uniform(*p['temp']),
            'humidity':    np.random.uniform(*p['hum']),
            'ph':          np.random.uniform(*p['ph']),
            'rainfall':    np.random.uniform(*p['rain']),
            'label':       crop
        })

df = pd.DataFrame(rows).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"✅ Dataset loaded: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"🌱 Unique crops   : {df['label'].nunique()}")
df.head(10)

## 🔍 Step 3 — Exploratory Data Analysis (EDA)

In [ ]:
# --- 3.1 Basic Info ---
print("=" * 55)
print("  DATASET OVERVIEW")
print("=" * 55)
print(df.info())
print("\nMissing Values:")
print(df.isnull().sum())

In [ ]:
# --- 3.2 Statistical Summary ---
df.describe().round(2)

In [ ]:
# --- 3.3 Class Distribution ---
fig, ax = plt.subplots(figsize=(14, 5))
crop_counts = df['label'].value_counts()
colors = plt.cm.tab20(np.linspace(0, 1, len(crop_counts)))
bars = ax.bar(crop_counts.index, crop_counts.values, color=colors, edgecolor='white', linewidth=0.8)
ax.set_title('🌾 Crop Class Distribution', fontsize=15, fontweight='bold', pad=15)
ax.set_xlabel('Crop Label', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.tick_params(axis='x', rotation=45)
for bar, count in zip(bars, crop_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            str(count), ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.show()
print(f"\n📊 Balanced dataset: {crop_counts.min()} – {crop_counts.max()} samples per class")

In [ ]:
# --- 3.4 Feature Distributions ---
features = ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()
palette = ['#2196F3','#4CAF50','#FF9800','#E91E63','#9C27B0','#00BCD4','#FF5722']

for i, feat in enumerate(features):
    axes[i].hist(df[feat], bins=40, color=palette[i], edgecolor='white', alpha=0.85)
    axes[i].set_title(f'{feat}', fontsize=12, fontweight='bold')
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Frequency')
    axes[i].axvline(df[feat].mean(), color='red', linestyle='--', linewidth=1.5, label=f'Mean={df[feat].mean():.1f}')
    axes[i].legend(fontsize=8)

axes[-1].axis('off')  # hide last empty subplot
fig.suptitle('📊 Feature Distributions', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# --- 3.5 Correlation Heatmap ---
fig, ax = plt.subplots(figsize=(9, 7))
corr = df[features].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn',
            mask=mask, linewidths=0.5, ax=ax,
            annot_kws={'size': 10})
ax.set_title('🔥 Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# --- 3.6 Boxplots: Feature by Crop (top crops) ---
top_crops = ['rice', 'maize', 'cotton', 'coffee', 'coconut', 'apple', 'banana', 'grapes']
df_top = df[df['label'].isin(top_crops)]

fig, axes = plt.subplots(2, 4, figsize=(20, 9))
axes = axes.flatten()
for i, feat in enumerate(features):
    sns.boxplot(data=df_top, x='label', y=feat, ax=axes[i],
                palette='Set2', width=0.6)
    axes[i].set_title(f'{feat} by Crop', fontsize=11, fontweight='bold')
    axes[i].set_xlabel('')
    axes[i].tick_params(axis='x', rotation=30)
axes[-1].axis('off')
fig.suptitle('📦 Feature Spread Across Selected Crops', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

## ⚙️ Step 4 — Data Preprocessing

In [ ]:
# --- 4.1 Encode Labels ---
le = LabelEncoder()
df['label_encoded'] = le.fit_transform(df['label'])
num_classes = len(le.classes_)
print(f"Number of classes : {num_classes}")
print(f"Class mapping     :")
for idx, crop in enumerate(le.classes_):
    print(f"  {idx:2d} → {crop}")

In [ ]:
# --- 4.2 Features & Target Split ---
X = df[features].values
y = df['label_encoded'].values
y_ohe = to_categorical(y, num_classes=num_classes)  # One-Hot Encoding

print(f"X shape : {X.shape}   (samples × features)")
print(f"y shape : {y_ohe.shape} (samples × classes)")

In [ ]:
# --- 4.3 Train / Validation / Test Split (70 / 15 / 15) ---
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y_ohe, test_size=0.30, random_state=42, stratify=y)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42,
    stratify=np.argmax(y_temp, axis=1))

print(f"Training   : {X_train.shape[0]} samples")
print(f"Validation : {X_val.shape[0]} samples")
print(f"Test       : {X_test.shape[0]} samples")

In [ ]:
# --- 4.4 Feature Scaling (StandardScaler on train, apply to val & test) ---
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_val_sc   = scaler.transform(X_val)
X_test_sc  = scaler.transform(X_test)

print("✅ Feature scaling applied")
print(f"   Train mean  : {X_train_sc.mean():.6f}  (should ≈ 0)")
print(f"   Train std   : {X_train_sc.std():.6f}   (should ≈ 1)")

## 🏗️ Step 5 — Build ANN Model

In [ ]:
# === ANN Architecture ===
#
#  Input (7) → Dense(128, ReLU) → BN → Dropout(0.3)
#            → Dense(256, ReLU) → BN → Dropout(0.3)
#            → Dense(128, ReLU) → BN → Dropout(0.2)
#            → Dense(64,  ReLU) → BN → Dropout(0.2)
#            → Output(22, Softmax)

def build_ann(input_dim, num_classes):
    model = keras.Sequential([
        # Input Layer
        layers.InputLayer(shape=(input_dim,)),

        # Hidden Layer 1
        layers.Dense(128, kernel_regularizer=regularizers.l2(1e-4)),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Dropout(0.3),

        # Hidden Layer 2
        layers.Dense(256, kernel_regularizer=regularizers.l2(1e-4)),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Dropout(0.3),

        # Hidden Layer 3
        layers.Dense(128, kernel_regularizer=regularizers.l2(1e-4)),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Dropout(0.2),

        # Hidden Layer 4
        layers.Dense(64, kernel_regularizer=regularizers.l2(1e-4)),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Dropout(0.2),

        # Output Layer
        layers.Dense(num_classes, activation='softmax')
    ], name='CropANN')
    return model

model = build_ann(input_dim=len(features), num_classes=num_classes)
model.summary()

In [ ]:
# --- Compile ---
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
print("✅ Model compiled")
print(f"   Total parameters: {model.count_params():,}")

## 🏋️ Step 6 — Train the Model

In [ ]:
# --- Callbacks ---
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=15,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=7,
        min_lr=1e-6,
        verbose=1
    ),
    ModelCheckpoint(
        filepath='best_crop_ann.keras',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=0
    )
]

# --- Train ---
history = model.fit(
    X_train_sc, y_train,
    epochs=150,
    batch_size=32,
    validation_data=(X_val_sc, y_val),
    callbacks=callbacks,
    verbose=1
)

## 📊 Step 7 — Training Curves

In [ ]:
def plot_history(history):
    epochs_ran = len(history.history['loss'])
    ep = range(1, epochs_ran + 1)

    fig, axes = plt.subplots(1, 2, figsize=(15, 5))

    # Loss
    axes[0].plot(ep, history.history['loss'],     color='#E91E63', linewidth=2, label='Train Loss')
    axes[0].plot(ep, history.history['val_loss'], color='#2196F3', linewidth=2, label='Val Loss',   linestyle='--')
    axes[0].set_title('📉 Loss Curve', fontsize=13, fontweight='bold')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
    axes[0].legend(); axes[0].grid(alpha=0.3)

    # Accuracy
    axes[1].plot(ep, history.history['accuracy'],     color='#4CAF50', linewidth=2, label='Train Acc')
    axes[1].plot(ep, history.history['val_accuracy'], color='#FF9800', linewidth=2, label='Val Acc', linestyle='--')
    axes[1].set_title('📈 Accuracy Curve', fontsize=13, fontweight='bold')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
    axes[1].set_ylim(0, 1.05)
    axes[1].legend(); axes[1].grid(alpha=0.3)

    best_epoch = np.argmax(history.history['val_accuracy']) + 1
    best_val_acc = max(history.history['val_accuracy'])
    axes[1].axvline(best_epoch, color='red', linestyle=':', linewidth=1.5,
                    label=f'Best @ epoch {best_epoch}')
    axes[1].legend()

    plt.suptitle('🏋️ Training History', fontsize=15, fontweight='bold')
    plt.tight_layout()
    plt.show()
    print(f"\n🏅 Best validation accuracy: {best_val_acc:.4f} at epoch {best_epoch}")

plot_history(history)

## 🧪 Step 8 — Evaluate on Test Set

In [ ]:
# --- Test Evaluation ---
test_loss, test_acc = model.evaluate(X_test_sc, y_test, verbose=0)
print("=" * 45)
print(f"  Test Loss     : {test_loss:.4f}")
print(f"  Test Accuracy : {test_acc * 100:.2f}%")
print("=" * 45)

In [ ]:
# --- Predictions ---
y_pred_proba = model.predict(X_test_sc, verbose=0)
y_pred       = np.argmax(y_pred_proba, axis=1)
y_true       = np.argmax(y_test,       axis=1)

# Classification Report
print("\n📋 Classification Report\n")
print(classification_report(y_true, y_pred,
                             target_names=le.classes_,
                             digits=4))

In [ ]:
# --- Confusion Matrix ---
cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(16, 14))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_)
disp.plot(ax=ax, colorbar=True, cmap='Blues', xticks_rotation=45)
ax.set_title('🔲 Confusion Matrix — ANN Crop Classification',
             fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

## 📊 Step 9 — Per-Class Performance Visualization

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

precision = precision_score(y_true, y_pred, average=None)
recall    = recall_score(y_true, y_pred, average=None)
f1        = f1_score(y_true, y_pred, average=None)

perf_df = pd.DataFrame({
    'Crop':      le.classes_,
    'Precision': precision,
    'Recall':    recall,
    'F1-Score':  f1
}).sort_values('F1-Score', ascending=True)

fig, ax = plt.subplots(figsize=(10, 10))
x = np.arange(len(perf_df))
width = 0.25

ax.barh(x - width, perf_df['Precision'], width, label='Precision', color='#2196F3', alpha=0.85)
ax.barh(x,         perf_df['Recall'],    width, label='Recall',    color='#4CAF50', alpha=0.85)
ax.barh(x + width, perf_df['F1-Score'],  width, label='F1-Score',  color='#FF9800', alpha=0.85)

ax.set_yticks(x)
ax.set_yticklabels(perf_df['Crop'], fontsize=10)
ax.set_xlabel('Score', fontsize=12)
ax.set_title('📊 Per-Class Precision / Recall / F1-Score', fontsize=14, fontweight='bold')
ax.legend(loc='lower right')
ax.set_xlim(0, 1.1)
ax.axvline(1.0, color='gray', linestyle='--', linewidth=0.8)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

## 🔮 Step 10 — Prediction Confidence Visualization

In [ ]:
# --- Top-5 Confidence Bar for sample predictions ---
sample_indices = np.random.choice(len(X_test_sc), size=6, replace=False)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, idx in enumerate(sample_indices):
    probs      = y_pred_proba[idx]
    top5_idx   = np.argsort(probs)[::-1][:5]
    top5_crops = le.inverse_transform(top5_idx)
    top5_probs = probs[top5_idx]
    true_label = le.inverse_transform([y_true[idx]])[0]
    pred_label = le.inverse_transform([y_pred[idx]])[0]

    colors = ['#4CAF50' if c == true_label else '#2196F3' for c in top5_crops]
    bars = axes[i].bar(top5_crops, top5_probs, color=colors, edgecolor='white')
    axes[i].set_title(
        f'True: {true_label} | Pred: {pred_label}',
        fontsize=10,
        color='green' if true_label == pred_label else 'red',
        fontweight='bold'
    )
    axes[i].set_ylim(0, 1.05)
    axes[i].set_ylabel('Confidence')
    axes[i].tick_params(axis='x', rotation=30)
    for bar, prob in zip(bars, top5_probs):
        axes[i].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                     f'{prob:.2f}', ha='center', va='bottom', fontsize=9)

correct_patch = mpatches.Patch(color='#4CAF50', label='True Class')
other_patch   = mpatches.Patch(color='#2196F3', label='Other')
fig.legend(handles=[correct_patch, other_patch], loc='upper right', fontsize=11)
fig.suptitle('🔮 Prediction Confidence — Top-5 Crops per Sample',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 🌿 Step 11 — Real-World Prediction Interface

In [ ]:
def predict_crop(N, P, K, temperature, humidity, ph, rainfall, top_k=3):
    """
    Predict the best crop to grow based on soil & climate inputs.
    Returns top-k crop recommendations with confidence.
    """
    sample = np.array([[N, P, K, temperature, humidity, ph, rainfall]])
    sample_sc = scaler.transform(sample)
    probs = model.predict(sample_sc, verbose=0)[0]
    top_k_idx   = np.argsort(probs)[::-1][:top_k]
    top_k_crops = le.inverse_transform(top_k_idx)

    print("=" * 50)
    print("  🌾 CROP RECOMMENDATION RESULTS")
    print("=" * 50)
    print(f"  Input Parameters:")
    print(f"  N={N}, P={P}, K={K}, Temp={temperature}°C")
    print(f"  Humidity={humidity}%, pH={ph}, Rainfall={rainfall}mm")
    print("-" * 50)
    print(f"  Top-{top_k} Recommendations:")
    for rank, (crop, prob) in enumerate(zip(top_k_crops, probs[top_k_idx]), 1):
        bar = '█' * int(prob * 30)
        print(f"  #{rank}  {crop:15s}  {prob*100:6.2f}%  {bar}")
    print("=" * 50)
    return top_k_crops[0]

# --- Test Case 1: Rice-like conditions ---
predict_crop(N=80, P=40, K=40, temperature=23, humidity=85, ph=6.5, rainfall=200)
print()
# --- Test Case 2: Coffee-like conditions ---
predict_crop(N=100, P=25, K=30, temperature=22, humidity=60, ph=5.0, rainfall=200)
print()
# --- Test Case 3: Cotton-like conditions ---
predict_crop(N=120, P=75, K=70, temperature=30, humidity=62, ph=7.0, rainfall=80)

## 📝 Step 12 — Summary & Conclusion

In [ ]:
# Final Summary Metrics
macro_f1  = f1_score(y_true, y_pred, average='macro')
micro_f1  = f1_score(y_true, y_pred, average='micro')
weighted_f1 = f1_score(y_true, y_pred, average='weighted')

print("=" * 50)
print("  📊 FINAL MODEL SUMMARY")
print("=" * 50)
print(f"  Architecture  : 4-Layer Deep ANN")
print(f"  Input Features: {len(features)}")
print(f"  Output Classes: {num_classes}")
print(f"  Parameters    : {model.count_params():,}")
print("-" * 50)
print(f"  Test Accuracy : {test_acc*100:.2f}%")
print(f"  Test Loss     : {test_loss:.4f}")
print(f"  Macro F1      : {macro_f1:.4f}")
print(f"  Micro F1      : {micro_f1:.4f}")
print(f"  Weighted F1   : {weighted_f1:.4f}")
print("=" * 50)
print("\n✅ Model trained and ready for deployment!")
print("💾 Best model saved as: best_crop_ann.keras")

---
## 🚀 Key Takeaways

| Aspect | Detail |
|--------|--------|
| **Dataset** | Crop Recommendation — 7 features, 22 crop classes |
| **Architecture** | 4 hidden layers (128 → 256 → 128 → 64) + BatchNorm + Dropout |
| **Regularization** | L2 weight decay + Dropout (0.2–0.3) |
| **Optimizer** | Adam (lr=1e-3) with ReduceLROnPlateau |
| **Loss Function** | Categorical Cross-Entropy |
| **Output** | Softmax (probability distribution over 22 crops) |
| **Callbacks** | EarlyStopping, ReduceLROnPlateau, ModelCheckpoint |

### 🌱 Real-World Applications
- 📱 **Mobile apps** for farmers to get instant crop suggestions
- 🌍 **Government portals** for precision agriculture policy
- 🤖 **IoT integration** with smart soil sensors for real-time recommendations